# Normalização de Texto

A sequência de código abaixo normaliza o texto mudando o encoding para ASCII e posteriormente retornando para UTF-8.

In [ ]:
import unicodedata

# texto original
original = "Café torrado e moído de graça feito pelo vovô e pela vovó."
print(original)

Café torrado e moído de graça feito pelo vovô e pela vovó.


Vamos utilizar a função normalize para decompor os caracteres. No caso, cada letra fica com um código separado do acento.

In [ ]:
nfd = unicodedata.normalize("NFD", original)
nfd

'Café torrado e moído de graça feito pelo vovô e pela vovó.'

Converte para ASCII, que é um padrão que possui apenas 128 caracteres comuns. O ignore é utilizado para que não seja apresentado erro caso haja um caractere que não haja correspondência em Ascii. Repare que após o encode, a cadeia de caracteres não é mais uma string, é uma sequência de bytes.

In [ ]:
ascii = nfd.encode("ascii", "ignore")
ascii

b'Cafe torrado e moido de graca feito pelo vovo e pela vovo.'

Vamos converter para UTF-8 para que fique novamente em formato de string.

In [ ]:
# Volta para UTF-8
utf8 = ascii.decode("utf-8")
utf8

'Cafe torrado e moido de graca feito pelo vovo e pela vovo.'

## Função de normalização
A função abaixo faz a normalização do texto de forma mais elegante e em um único passo, mantendo o encoding correto. Também transforma o texto em maiúsculas.

In [ ]:
def padroniza_str(str):
  nfd = unicodedata.
  return nfd.encode("ascii", "ignore").decode("utf-8").upper()

In [ ]:
print(original)
print(padroniza_str(original))

Café torrado e moído de graça feito pelo vovô e pela vovó.
CAFE TORRADO E MOIDO DE GRACA FEITO PELO VOVO E PELA VOVO.


## Vamos incrementar o pipeline ETL construido no NB2 com a normalização de texto.
Vamos começar do nosso arquivo STG.

In [ ]:
import pandas as pd
import os
from google.colab import drive

In [ ]:
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/Docencia/IDP/Dados'
df_classe = pd.read_parquet(os.path.join(DATA_PATH, 'stg_classe.parquet'))

Mounted at /content/drive


##Vamos implementar as alterações feitas anteriormente.
1. Atualização da data para datetime.

In [ ]:
df_classe['dataHoraAtualizacao'] = pd.to_datetime(df_classe['dataHoraAtualizacao'])
display(df_classe.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 711 entries, 0 to 710
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   codigoClasse         711 non-null    int64         
 1   codigoGrupo          711 non-null    int64         
 2   nomeGrupo            711 non-null    object        
 3   nomeClasse           711 non-null    object        
 4   statusClasse         711 non-null    bool          
 5   dataHoraAtualizacao  711 non-null    datetime64[ns]
dtypes: bool(1), datetime64[ns](1), int64(2), object(2)
memory usage: 28.6+ KB


None

2. Criar a coluna com a data da carga.

In [ ]:
df_classe['data_carga'] = pd.to_datetime('today').normalize()
display(df_classe.head())
display(df_classe.info())

,codigoClasse,codigoGrupo,nomeGrupo,nomeClasse,statusClasse,dataHoraAtualizacao,data_carga
0,1005,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ATÉ 120MM,True,2021-10-16 09:17:13.045775,2026-08-08
1,1010,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 30MM ATÉ 75MM,True,2021-10-16 09:17:13.045775,2026-08-08
2,1015,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 75MM ATÉ 125MM,True,2021-10-16 09:17:13.045775,2026-08-08
3,1020,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 125MM ATÉ 150MM,True,2021-10-16 09:17:13.045775,2026-08-08
4,1025,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 150MM ATÉ 200MM,True,2021-10-16 09:17:13.045775,2026-08-08


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 711 entries, 0 to 710
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   codigoClasse         711 non-null    int64         
 1   codigoGrupo          711 non-null    int64         
 2   nomeGrupo            711 non-null    object        
 3   nomeClasse           711 non-null    object        
 4   statusClasse         711 non-null    bool          
 5   dataHoraAtualizacao  711 non-null    datetime64[ns]
 6   data_carga           711 non-null    datetime64[ns]
dtypes: bool(1), datetime64[ns](2), int64(2), object(2)
memory usage: 34.2+ KB


None

3. Agora vamos normalizar a a coluna nomeClasse

In [ ]:
df_classe['nomeClasse'] = df_classe['nomeClasse'].apply(padroniza_str)
display(df_classe.head())

,codigoClasse,codigoGrupo,nomeGrupo,nomeClasse,statusClasse,dataHoraAtualizacao,data_carga
0,1005,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ATE 120MM,True,2021-10-16 09:17:13.045775,2026-08-08
1,1010,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 30MM ATE 75MM,True,2021-10-16 09:17:13.045775,2026-08-08
2,1015,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 75MM ATE 125MM,True,2021-10-16 09:17:13.045775,2026-08-08
3,1020,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 125MM ATE 150MM,True,2021-10-16 09:17:13.045775,2026-08-08
4,1025,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 150MM ATE 200MM,True,2021-10-16 09:17:13.045775,2026-08-08


4. Por fim, salvar a nova versão do parquet definitivo

In [ ]:
df_classe.to_parquet(os.path.join(DATA_PATH, 'classe.parquet'))
print(f"DataFrame df_classe salvo como 'classe.parquet' em {DATA_PATH}")

DataFrame df_classe salvo como 'classe.parquet' em /content/drive/MyDrive/Docencia/IDP/Dados
